In [2]:
import json
import re
import pickle
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from transformers import AutoTokenizer, AutoModel
from rapidfuzz import fuzz

c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
DATA_PATH = "dataset_min4.json"
MODEL_NAME = "DeepPavlov/rubert-base-cased"
SAVE_PATH = "bert_index_classifiers_final.pkl"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

cpu


In [4]:
with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

print("Всего примеров:", len(df))
print("Всего index:", df["index"].nunique())

Всего примеров: 6578
Всего index: 260


In [5]:
def extract_phrase(example):
    match = re.search(r"=(.*?)=", example)
    if match:
        return match.group(1).lower().strip()
    return None


index_to_phrases = {}

for index_value, group in df.groupby("index"):
    phrases = group["example"].apply(extract_phrase).dropna().tolist()
    phrases = sorted(set(phrases), key=len, reverse=True)
    index_to_phrases[index_value] = phrases

print(index_to_phrases)

{1: ['без году неделя', 'без года неделя'], 2: ['без задних ног'], 3: ['без конца и без краю', 'без конца и без края', 'без конца, без края', 'без конца и края', 'без конца и краю'], 4: ['без сучка и без задоринки', 'без сучка, без задоринки', 'без сучка без задоринки', 'без сучка и задоринки'], 5: ['благим матом'], 6: ['бог ведает', 'бог знает', 'бог весть'], 7: ['бок о бок с первым украинским фронтом', 'бок о бок с бочонком бензина', 'бок о бок с хитрыми станками', 'о бок с его существованием', 'бок о бок с рекою', 'бок о бок'], 8: ['большой руки'], 9: ['взять тебя (таис. — авт.) в руки', 'взял власть в городе в свои руки', 'взяв его в свои энергичные руки', 'взял руководство в свои руки', 'взяли в свои руки гвельфы', 'забрать весь дом в руки', 'возьмите себя в руки', 'возьму себя в руки', 'брать себя в руки', 'берет себя в руки', 'взяла себя в руки', 'брала себя в руки', 'взять себя в руки', 'брал себя в руки', 'взял себя в руки', 'беру себя в руки'], 11: ['набрасывает неприятную те

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
bert.eval()


def get_bert_embeddings(texts, batch_size=16, max_length=128):
    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc="BERT embeddings"):
        batch_texts = texts[i:i + batch_size]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(DEVICE)

        with torch.no_grad():
            outputs = bert(**encoded)

        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        all_embeddings.append(cls_embeddings.cpu().numpy())

    return np.vstack(all_embeddings)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7852.43it/s]
[transformers] BertModel LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
train_parts = []
test_parts = []

for index_value, group in df.groupby("index"):
    train_g, test_g = train_test_split(
        group,
        test_size=0.25,
        random_state=42,
        stratify=group["sense_number"]
    )

    train_parts.append(train_g)
    test_parts.append(test_g)

train_df = pd.concat(train_parts).reset_index(drop=True)
test_df = pd.concat(test_parts).reset_index(drop=True)

print("Train:", len(train_df))
print("Test:", len(test_df))

Train: 4832
Test: 1746


In [8]:
# X_train = get_bert_embeddings(train_df["example"].tolist())
# X_test = get_bert_embeddings(test_df["example"].tolist())

# np.save("X_train.npy", X_train)
# np.save("X_test.npy", X_test)

# train_df.to_json("train_df.json", orient="records", force_ascii=False, indent=2)
# test_df.to_json("test_df.json", orient="records", force_ascii=False, indent=2)

In [9]:
X_train = np.load("X_train.npy")
X_test = np.load("X_test.npy")

train_df = pd.read_json("train_df.json")
test_df = pd.read_json("test_df.json")

In [10]:
classifiers = {}
baselines = {}
meaning_map = {}

results = []

for index_value in sorted(train_df["index"].unique()):
    train_mask = train_df["index"] == index_value
    test_mask = test_df["index"] == index_value

    X_train_i = X_train[train_mask.values]
    y_train_i = train_df.loc[train_mask, "sense_number"].values

    X_test_i = X_test[test_mask.values]
    y_test_i = test_df.loc[test_mask, "sense_number"].values

    meaning_map[index_value] = (
        train_df[train_df["index"] == index_value]
        .drop_duplicates(["sense_number"])
        .set_index("sense_number")["meaning"]
        .to_dict()
    )

    most_common_sense = Counter(y_train_i).most_common(1)[0][0]
    baselines[index_value] = most_common_sense

    baseline_preds = [most_common_sense] * len(y_test_i)
    baseline_acc = accuracy_score(y_test_i, baseline_preds)

    clf = LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    )

    clf.fit(X_train_i, y_train_i)
    pred_i = clf.predict(X_test_i)

    model_acc = accuracy_score(y_test_i, pred_i)

    classifiers[index_value] = clf

    results.append({
        "index": index_value,
        "baseline_acc": baseline_acc,
        "bert_classifier_acc": model_acc,
        "n_train": len(y_train_i),
        "n_test": len(y_test_i),
        "n_senses": len(set(y_train_i))
    })


results_df = pd.DataFrame(results)

In [11]:
def find_index_by_phrase(example, threshold=80):
    example_lower = example.lower().replace("=", "")

    best_index = None
    best_phrase = None
    best_score = 0

    for index_value, phrases in index_to_phrases.items():
        for phrase in phrases:
            score = fuzz.partial_ratio(phrase, example_lower)

            if score > best_score:
                best_score = score
                best_index = index_value
                best_phrase = phrase

    if best_score >= threshold:
        return best_index, best_phrase

    return None, None

In [12]:
def predict_full(example):
    index_value, found_phrase = find_index_by_phrase(example)

    if index_value is None:
        return {
            "error": "Фразеологизм не найден"
        }

    emb = get_bert_embeddings([example])

    clf = classifiers[index_value]
    sense_number = int(clf.predict(emb)[0])

    meaning = meaning_map[index_value].get(sense_number, "meaning не найден")

    return {
        "found_phrase": found_phrase,
        "index": index_value,
        "sense_number": sense_number,
        "meaning": meaning
    }

In [13]:
full_results = []

for i, row in test_df.reset_index(drop=True).iterrows():
    example = row["example"]
    true_index = row["index"]
    true_sense = row["sense_number"]

    index_value, found_phrase = find_index_by_phrase(example)

    if index_value is None:
        pred_index = None
        pred_sense = None
        correct_index = False
        correct_sense = False
    else:
        pred_index = index_value

        emb = X_test[i].reshape(1, -1)

        clf = classifiers[index_value]
        pred_sense = int(clf.predict(emb)[0])

        correct_index = pred_index == true_index
        correct_sense = (pred_index == true_index) and (pred_sense == true_sense)

    full_results.append({
        "example": example,
        "found_phrase": found_phrase,
        "true_index": true_index,
        "pred_index": pred_index,
        "true_sense": true_sense,
        "pred_sense": pred_sense,
        "correct_index": correct_index,
        "correct_sense": correct_sense
    })

full_results_df = pd.DataFrame(full_results)

bert_errors = full_results_df[
    full_results_df["correct_sense"] == False
]

bert_errors.to_csv("bert_errors.csv", index=False, encoding="utf-8-sig")

In [14]:
print("МЕТРИКИ")
print("Index accuracy:", full_results_df["correct_index"].mean())
print("Sense accuracy:", full_results_df["correct_sense"].mean())
print("All accuracy:", (full_results_df["correct_index"] & full_results_df["correct_sense"]).mean())

МЕТРИКИ
Index accuracy: 0.9501718213058419
Sense accuracy: 0.6592210767468499
All accuracy: 0.6592210767468499


In [15]:
test_example = "=Божья коровка=, улети на небо, принеси нам хлеба..."

print("Пример предсказания")
print(predict_full(test_example))

Пример предсказания


BERT embeddings:   0%|          | 0/1 [00:00<?, ?it/s]

BERT embeddings: 100%|██████████| 1/1 [00:01<00:00,  1.99s/it]

{'found_phrase': 'божья коровка', 'index': 227, 'sense_number': 1, 'meaning': 'семейство жуков длиной 4—7 мм красной, желтой или белой окраски с черными пятнами, истребляющее тлю и других вредных насекомых'}
